# CPU • TorchScript

ResNet-18, 50 images du split de test CIFAR-10 et lots de 10. Trois exécutions dans des processus Python distincts.

**Runtime Colab** · CPU.
Exécuter d’abord le notebook 03 pour préparer les artefacts communs.

In [1]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/TP1_ResNet18")
ROOT.mkdir(parents=True, exist_ok=True)
CAMPAIGN = "colab-tp1"  # Même valeur dans les quatre notebooks.
ENV = dict(os.environ, OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")
SCRIPT = Path("/content/tp1_benchmark.py")

def executer(*args):
    subprocess.run([sys.executable, str(SCRIPT), str(ROOT), *map(str, args)],
                   env=ENV, check=True)

Mounted at /content/drive


## Implémentation du pipeline

        Les trois processus rechargent le modèle et exécutent le même protocole. Les fichiers JSON
        contiennent les mesures, les prédictions et la configuration matérielle.

In [2]:
%%writefile /content/tp1_benchmark.py
import argparse
import hashlib
import json
import platform
import re
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
import torchvision
from torchvision.models import resnet18

BATCH = 10
THREADS = 2
WARMUP = 5
PASSES = 10
RUNS = 3
DIGESTS = ("weights.pt", "test_data.pt", "model.onnx")
SCOPE = "Entrée CPU prétraitée → logits CPU, transferts inclus"


def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def write_json(path, data):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    tmp.replace(path)


def torchscript_predictor(root, gpu):
    device = "cuda" if gpu else "cpu"
    model = resnet18(weights=None, num_classes=10).eval()
    model.load_state_dict(torch.load(root / "weights.pt", map_location="cpu", weights_only=True))
    with torch.no_grad():
        model = model.to(device)
        traced = torch.jit.freeze(torch.jit.trace(model, torch.zeros(BATCH, 3, 224, 224, device=device)))

    def predict(batch):
        return traced(batch.to(device)).cpu().numpy()
    return predict, str(torch.__version__)


def energy_reader(gpu):
    if gpu:
        try:
            import pynvml as nv
        except ImportError:
            return None, None
        try:
            nv.nvmlInit()
            props = torch.cuda.get_device_properties(0)
            uuid = getattr(props, "uuid", None)
            if nv.nvmlDeviceGetCount() == 1:
                handle = nv.nvmlDeviceGetHandleByIndex(0)
            elif uuid is not None:
                handle = nv.nvmlDeviceGetHandleByUUID(str(uuid))
            else:
                return None, None
            reader = lambda: nv.nvmlDeviceGetTotalEnergyConsumption(handle) / 1e3
            reader()
            return "GPU NVML", reader
        except (OSError, nv.NVMLError):
            return None, None
    zones = list(Path("/sys/class/powercap").glob("intel-rapl:[0-9]*"))
    try:
        packages = [p for p in zones if re.fullmatch(r"intel-rapl:\d+", p.name)
                    and (p / "name").read_text().strip().startswith("package")]
        if len(packages) == 1:
            package = packages[0]
            reader = lambda: int((package / "energy_uj").read_text()) / 1e6
            reader()
            return "Package CPU RAPL", reader
    except (OSError, ValueError):
        pass
    return None, None


def energy_value(reader):
    try:
        return reader() if reader else None
    except Exception:
        # Une erreur du compteur ne doit pas faire perdre les mesures de latence.
        return None


def cpu_name():
    if Path("/proc/cpuinfo").exists():
        match = re.search(r"^model name\s*:\s*(.+)$", Path("/proc/cpuinfo").read_text(), re.M)
        if match:
            return match.group(1)
    return platform.processor() or platform.machine()


def benchmark(root, pipeline, run, campaign, colab, launch_group):
    torch.set_num_threads(THREADS)
    torch.set_num_interop_threads(1)
    gpu = pipeline.startswith("gpu_")
    if gpu:
        if not torch.cuda.is_available():
            raise RuntimeError("Activer un GPU NVIDIA dans Colab.")
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
    manifest = json.loads((root / "manifest.json").read_text())
    for name in DIGESTS:
        if sha256(root / name) != manifest["files"][name]:
            raise RuntimeError(f"Fichier commun modifié ou incomplet : {name}")
    data = torch.load(root / "test_data.pt", map_location="cpu", weights_only=True)
    if data["X"].shape != (50, 3, 224, 224) or data["Y"].shape != (50,):
        raise RuntimeError("Les données d’évaluation doivent contenir 50 images.")
    batches, labels = data["X"].split(BATCH), data["Y"].numpy()
    predict, version = torchscript_predictor(root, gpu)
    source, reader = energy_reader(gpu)
    with torch.inference_mode():
        for i in range(WARMUP):
            predict(batches[i % len(batches)])
        if gpu:
            torch.cuda.synchronize()
        durations = []
        started_energy = energy_value(reader)
        expected = None
        for _ in range(PASSES):
            outputs = []
            for batch in batches:
                start = perf_counter()
                logits = predict(batch)
                durations.append(perf_counter() - start)
                outputs.append(logits)
            logits = np.concatenate(outputs)
            if not np.isfinite(logits).all():
                raise RuntimeError("Logits non finis.")
            predictions = logits.argmax(axis=1)
            if expected is None:
                expected = predictions
            elif not np.array_equal(predictions, expected):
                raise RuntimeError("Prédictions variables pendant cette exécution.")
        ended_energy = energy_value(reader)
    energy = None
    if started_energy is not None and ended_energy is not None and ended_energy > started_energy:
        energy = ended_energy - started_energy
    count = len(labels) * PASSES
    result = {
        "pipeline": pipeline, "run": run, "campaign": campaign, "colab": colab,
        "launch_group": launch_group,
        "manifest": manifest["files"], "indices": data["indices"],
        "labels": labels.tolist(), "predictions": expected.tolist(),
        "metrics": {"debit_images_s": count / sum(durations),
                    "latence_ms_lot": 1000 * float(np.mean(durations)),
                    "top1_pct": 100 * float(np.mean(expected == labels)),
                    "energie_j": energy, "energie_j_image": energy / count if energy else None},
        "metadata": {"cpu": cpu_name(), "gpu": torch.cuda.get_device_name(0) if gpu else None,
                     "python": platform.python_version(), "torch": str(torch.__version__),
                     "torchvision": torchvision.__version__, "runtime": version,
                     "cuda": torch.version.cuda if gpu else None,
                     "cudnn": torch.backends.cudnn.version() if gpu else None,
                     "threads": THREADS, "batch": BATCH, "passes": PASSES, "warmup": WARMUP,
                     "precision": "FP32 (TF32 désactivé sur GPU)", "scope": SCOPE,
                     "energy_sensor": source}
    }
    write_json(root / "mesures" / campaign / f"{pipeline}_{run}.json", result)
    print(pipeline, run, result["metrics"], flush=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("root", type=Path)
    parser.add_argument("pipeline", choices=('cpu_torchscript',))
    parser.add_argument("--run", type=int, choices=range(1, RUNS + 1))
    parser.add_argument("--campaign", default="colab-tp1")
    parser.add_argument("--colab", action="store_true")
    parser.add_argument("--groupe", default=None)
    args = parser.parse_args()
    if args.run is None or not re.fullmatch(r"[A-Za-z0-9_-]+", args.campaign):
        parser.error("Indiquer un numéro d’exécution et un nom de campagne valide.")
    (args.root / "mesures" / args.campaign).mkdir(parents=True, exist_ok=True)
    benchmark(args.root, args.pipeline, args.run, args.campaign, args.colab, args.groupe)

Writing /content/tp1_benchmark.py


## Mesures et statistiques

In [3]:
import json
import uuid
import pandas as pd

if not (ROOT / "manifest.json").exists():
    raise FileNotFoundError("Exécuter d’abord la préparation avec le notebook 03.")
PIPELINE = 'cpu_torchscript'
inputs = ROOT / "mesures" / CAMPAIGN
groupe = uuid.uuid4().hex
for numero in range(1, 4):
    executer(PIPELINE, "--campaign", CAMPAIGN, "--run", numero, "--colab", "--groupe", groupe)
mesures = [json.loads((inputs / f"{PIPELINE}_{i}.json").read_text()) for i in range(1, 4)]
df = pd.DataFrame([r["metrics"] for r in mesures], dtype=float)
print("Environnement :", mesures[-1]["metadata"])
display(df.rename_axis("exécution").set_axis(range(1, 4)))
display(df.agg(["count", "mean", "std", "min", "max"]))

Environnement : {'cpu': 'Intel(R) Xeon(R) CPU @ 2.20GHz', 'gpu': None, 'python': '3.13.18', 'torch': '2.14.1+cpu', 'torchvision': '0.29.1+cpu', 'runtime': '2.14.1+cpu', 'cuda': None, 'cudnn': None, 'threads': 2, 'batch': 10, 'passes': 10, 'warmup': 5, 'precision': 'FP32 (TF32 désactivé sur GPU)', 'scope': 'Entrée CPU prétraitée → logits CPU, transferts inclus', 'energy_sensor': None}


,debit_images_s,latence_ms_lot,top1_pct,energie_j,energie_j_image
1,11.255484,888.455830,76.0,NaN,NaN
2,11.741034,851.713752,76.0,NaN,NaN
3,12.292647,813.494444,76.0,NaN,NaN


,debit_images_s,latence_ms_lot,top1_pct,energie_j,energie_j_image
count,3.000000,3.000000,3.0,0.0,0.0
mean,11.763055,851.221342,76.0,NaN,NaN
std,0.518932,37.483119,0.0,NaN,NaN
min,11.255484,813.494444,76.0,NaN,NaN
max,12.292647,888.455830,76.0,NaN,NaN
